# 04 — Data Validation, Deterministic Splitting & Test Split Lock
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook enforces strict experimental integrity:
1. Validates schema constraints and data quality rules.
2. Splits dataset deterministically: **80% Train**, **10% Validation**, **10% Test** (Seed: 42).
3. Verifies ZERO data leakage across splits ($Train \cap Val = 0, Train \cap Test = 0, Val \cap Test = 0$).
4. **PROGRAMMATICALLY LOCKS `test.jsonl`**: Enforces that `test.jsonl` cannot be accessed by Notebooks 01–07.


In [ ]:
# Cell 1: Environment Setup & Splitting
import os
import json
import random
from pathlib import Path
from test_access_guard import verify_no_split_leakage, lock_test_dataset

WORKSPACE_DIR = Path(os.getcwd())
PREPROCESSED_FILE = WORKSPACE_DIR / "dataset" / "preprocessed" / "preprocessed_dataset.json"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

with open(PREPROCESSED_FILE, "r", encoding="utf-8") as f:
    records = json.load(f)

# Deterministic shuffling with fixed seed
random.seed(42)
shuffled = list(records)
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_recs = shuffled[:n_train]
val_recs = shuffled[n_train:n_train + n_val]
test_recs = shuffled[n_train + n_val:]

print(f"Splits Created: Train={len(train_recs)}, Validation={len(val_recs)}, Test={len(test_recs)}")


In [ ]:
# Cell 2: Leakage Check & File Export
# Assert zero overlap
leakage_report = verify_no_split_leakage(train_recs, val_recs, test_recs)
assert leakage_report["zero_leakage_passed"], "Data leakage detected across splits!"
print("Zero Data Leakage Assertion: PASSED")

# Save split files
for split_name, recs in [("train", train_recs), ("validation", val_recs), ("test", test_recs)]:
    s_dir = PROCESSED_DIR / split_name
    s_dir.mkdir(parents=True, exist_ok=True)
    out_file = s_dir / f"{split_name}.jsonl"
    with open(out_file, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {split_name} split to: {out_file} ({len(recs)} records)")

# Activate Test Lock Guard
lock_test_dataset(PROCESSED_DIR / "test")
print("TEST SPLIT HAS BEEN PROGRAMMATICALLY LOCKED.")
print("Stage 04 Completed Successfully.")
